**📖 Tutorial: Automated Kinetic State Discovery with PySoftK**
==================================================================

Overview: This notebook demonstrates how to process a raw Molecular Dynamics trajectory (like those generated by xTB), automatically identify the molecular backbone, filter out thermal noise, and cluster the distinct kinetic states using tICA and HDBSCAN. Finally, we will mathematically analyze and interactively visualize the discovered conformations.
Step 1: Install Necessary Libraries

First, we need to install the required Python libraries, RDKit, and PySoftK. This installation cell handles the process by cloning the pysoftk repository and then using pip install . to install the package and all its dependencies as defined in its setup.py file. We also install py3Dmol for interactive visualization in the browser.\

In [5]:
# Clone the pysoftk repository from GitHub
!git clone https://github.com/alejandrosantanabonilla/pysoftk

# Navigate to the pysoftk directory
%cd pysoftk

# Install the package from the current directory
!pip install .

# Install py3Dmol for in-browser 3D visualization
!pip install py3Dmol

Cloning into 'pysoftk'...
remote: Enumerating objects: 6403, done.
remote: Counting objects: 100% (341/341), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 6403 (delta 285), reused 224 (delta 206), pack-reused 6062 (from 4)
Receiving objects: 100% (6403/6403), 118.86 MiB | 33.07 MiB/s, done.
Resolving deltas: 100% (3803/3803), done.
Updating files: 100% (354/354), done.
/content/pysoftk/pysoftk/pysoftk/pysoftk/pysoftk
Processing /content/pysoftk/pysoftk/pysoftk/pysoftk/pysoftk
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/pyscf/semiempirical to /tmp/pip-install-fn5luctp/pyscf-semiempirical_92868f802cbd42bb9eb5afe462942b6b
  Running command git clone --filter=blob:none --quiet https://github.com/pyscf/semiempirical /tmp/pip-install-fn5luctp/pyscf-semiempirical_92868f802cbd42bb9eb5afe462942b6b
  Resolved https://github.com/pyscf/semiempirical to co

**Step 2: Define Data Paths**
=================================

For this tutorial, the required trajectory (xtb.xyz) and topology (topology.pdb) of Alanine Dipeptide are provided in the repository's data folder. Let's define the paths so our pipeline can find them.

In [7]:
import os

# Updated path to point directly to your specific data directory
data_dir = "topologies_tutorials/data"

topology_file = os.path.join(data_dir, "ala.pdb")
trajectory_file = os.path.join(data_dir, "xtb.xyz")
output_conformers = os.path.join(data_dir, "representative_conformers.pdb")

# Quick check to ensure the files are actually there!
if not os.path.exists(topology_file):
    print(f"⚠️ Warning: Could not find {topology_file}. Check your folder structure!")
else:
    print(f"✅ Topology found at: {topology_file}")

if not os.path.exists(trajectory_file):
    print(f"⚠️ Warning: Could not find {trajectory_file}. Check your folder structure!")
else:
    print(f"✅ Trajectory found at: {trajectory_file}")

✅ Topology found at: topologies_tutorials/data/ala.pdb
✅ Trajectory found at: topologies_tutorials/data/xtb.xyz


**Step 3: Run the Kinetic Clustering Pipeline**
=====================================================

Now we will use PySoftK to automatically extract the dominant conformations. The algorithm will use RDKit to find the structural backbone, build sparse spatial contact maps, project the data onto a slow kinetic manifold using tICA, and cluster the states using HDBSCAN.

In [8]:
from pysoftk.pol_analysis.kinetic_clustering import AutomatedKineticClustering

# 1. Initialize the pipeline
print("Loading trajectory data...")
app = AutomatedKineticClustering(topology_file, trajectory_file)

# 2. Identify the structural core (Top 60% centrality, filtering out fast-moving hydrogens)
app.define_backbone_by_centrality(percentile=60)

# 3. Build contact maps (4.5 Å cutoff for H-bonding in small peptides)
app.extract_contact_maps(r_cutoff=4.5)

# 4. Project and cluster (Lag time of 5 frames)
embedding, labels = app.run_tica_clustering(lag_time_frames=5, n_components=2, min_cs=10)

# 5. Save the physical frames closest to the cluster centers
app.export_representative_states(output_conformers)

Loading trajectory data...
Converting MDAnalysis topology to RDKit molecule...
Calculating RDKit topological distance matrix (C++ optimized)...
Reduced 22 total atoms down to a core backbone of 9 atoms (Top 40%).
Function define_backbone_by_centrality Took 0.0061 seconds
Extracting contact maps with r_cutoff = 4.5 Å...


/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Building Contact Maps:   0%|          | 0/1042 [00:00<?, ?it/s]

Function extract_contact_maps Took 0.1878 seconds
Running tICA. Lag time: 5 frames (5.0 ps)...
Clustering kinetic states with HDBSCAN...
Identified 3 distinct metastable states.
Frames in transition states (noise): 0 (0.00%)
Function run_tica_clustering Took 0.0489 seconds
State 0 -> Saved Frame 0 to topologies_tutorials/data/representative_conformers.pdb
State 1 -> Saved Frame 13 to topologies_tutorials/data/representative_conformers.pdb
State 2 -> Saved Frame 51 to topologies_tutorials/data/representative_conformers.pdb
Export complete.
Function export_representative_states Took 0.0077 seconds


/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/PDB.py:1336: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn(


**Step 4: Mathematical Analysis of Discovered States**
===========================================================

To prove the algorithm worked, we can use MDAnalysis to measure the Ramachandran dihedral angles (ϕ and ψ) and the critical intramolecular hydrogen bond distance. This will mathematically classify our discovered states as the β-sheet, PII​, or αR​ folded basins.

In [10]:
import numpy as np
import MDAnalysis as mda
from MDAnalysis.lib.distances import calc_dihedrals

print(f"--- Analyzing {output_conformers} ---")
u = mda.Universe(output_conformers)

# Atom indices for Alanine Dipeptide (0-based)
phi_indices = [4, 6, 8, 14] # C(ACE) - N - CA - C
psi_indices = [6, 8, 14, 16] # N - CA - C - N(NME)
o_idx, h_idx = 5, 17 # O(ACE) to H(NME) for H-Bond

for ts in u.trajectory:
    # Extract coordinates
    phi_coords = u.atoms[phi_indices].positions
    psi_coords = u.atoms[psi_indices].positions

    # Calculate Dihedrals
    phi_angle = np.rad2deg(calc_dihedrals(*phi_coords))
    psi_angle = np.rad2deg(calc_dihedrals(*psi_coords))

    # Calculate H-Bond Distance
    hbond_dist = np.linalg.norm(u.atoms[o_idx].position - u.atoms[h_idx].position)

    if hbond_dist < 2.5:
        state_name = "Alpha-Helix (aR / C7eq)"
    elif phi_angle < -130 and psi_angle > 100:
        # Fully extended
        state_name = "Beta-Sheet (C5)"
    else:
        # Twisted extended
        state_name = "Polyproline II (PII)"

    print(f"State {ts.frame + 1}: {state_name}")
    print(f"  -> Phi (φ): {phi_angle:.1f}°")
    print(f"  -> Psi (ψ): {psi_angle:.1f}°")
    print(f"  -> O-H Distance: {hbond_dist:.2f} Å\n")

--- Analyzing topologies_tutorials/data/representative_conformers.pdb ---
State 1: Beta-Sheet (C5)
  -> Phi (φ): -163.3°
  -> Psi (ψ): 168.0°
  -> O-H Distance: 5.02 Å

State 2: Polyproline II (PII)
  -> Phi (φ): -112.0°
  -> Psi (ψ): 163.2°
  -> O-H Distance: 4.48 Å

State 3: Alpha-Helix (aR / C7eq)
  -> Phi (φ): -84.9°
  -> Psi (ψ): 43.6°
  -> O-H Distance: 2.05 Å



**Step 5: Interactive 3D Visualization**
=============================================

Because Google Colab runs in a browser, standard PyMOL windows cannot open. Instead, we use py3Dmol to load the PDB file and visualize the extended and folded states right here in the notebook. You can click and drag to rotate the molecule!

In [11]:
import py3Dmol

print(f"--- Visualizing {output_conformers} ---")

# Read the generated PDB file containing our 3 states
with open(output_conformers, 'r') as f:
    pdb_data = f.read()

# Initialize the 3D viewer
view = py3Dmol.view(width=800, height=400)
view.addModelsAsFrames(pdb_data)

# Style the molecule (sticks and small spheres look great for peptides)
view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.4}})

# Animate through the states to see the molecule fold and unfold!
# 'step': 1000 means it will pause for 1000 milliseconds (1 second) on each state
view.animate({'loop': 'forward', 'step': 1000})
view.zoomTo()
view.show()

--- Visualizing topologies_tutorials/data/representative_conformers.pdb ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.